# Lab 4c: Evaluating an Object Detection Model

This is **Part 3 of 3** in a mini-series on evaluating deep learning models.

As in Lab 4b, **you will not train anything here** — we use a pretrained object detector and focus
entirely on evaluating it properly. Object detection evaluation is more involved than classification
or segmentation, because a prediction is only "correct" if *both* the predicted box location and the
predicted class are right — this is exactly why a dedicated metric, **mean Average Precision (mAP)**,
was developed for it.

## Your task

1. Build a small object detection dataset from the raw Oxford-IIIT Pet annotation files (bounding boxes).
2. Load a pretrained object detector and run it on the dataset.
3. Implement Intersection over Union (IoU) yourself, and use it to inspect individual predictions.
4. Evaluate the model with **mean Average Precision (mAP)**, and visualize some predictions.


In [ ]:
# Setup
!pip install -q torchmetrics pycocotools


In [ ]:
import torch
from torch.utils.data import Dataset

import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.utils import draw_bounding_boxes
from torchvision.transforms.functional import to_pil_image, pil_to_tensor

from torchmetrics.detection.mean_ap import MeanAveragePrecision

import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# The two classes we care about, using the *detector's* label indices (COCO's 91-class list):
CAT_ID, DOG_ID = 17, 18


## Task 1: Building a detection dataset from raw files

`torchvision.datasets.OxfordIIITPet` (which we used in Lab 4b) only exposes classification and
segmentation targets — it does not give us bounding boxes directly. The raw dataset download does
also include a separate set of bounding-box annotations, as one Pascal-VOC-style XML file per image
under `annotations/xmls/`, e.g.:

```xml
<annotation>
  <object>
    <name>cat</name>
    <bndbox>
      <xmin>65</xmin><ymin>25</ymin><xmax>318</xmax><ymax>294</ymax>
    </bndbox>
  </object>
</annotation>
```

**Watch out:** those XML boxes mark only the animal's **head/face**, not its whole body — that was
this dataset's own annotation convention. Our detector below, however, is a general-purpose model
pretrained on COCO, where "cat"/"dog" boxes always cover the *whole animal*. Comparing COCO-style
whole-body predictions against head-only ground truth would make IoU/mAP look artificially poor no
matter how good the detector is, so we don't use those XML files here.

Instead, we build our ground-truth boxes the same way we built ground-truth masks in Lab 4b: from
the **segmentation trimap**, which is available for every image in both splits and already outlines
the whole animal. Since a trimap already tells us exactly which pixels belong to the pet (label `1`)
or its border (label `3`), the tightest box containing all of those pixels *is* a whole-animal box —
with no extra download needed beyond what Lab 4b already used.

The `Dataset` class below is given in full (it's a thin wrapper around the segmentation dataset from
Lab 4b, not a new deep learning concept) — but it's a useful example of how you can adapt an existing
dataset into a different target format for your own project.


In [ ]:
class OxfordPetDetection(Dataset):
    '''Wraps Lab 4b's segmentation dataset to expose a whole-animal bounding box per image: the
    tight box around all foreground (pet + border) pixels in the trimap.'''

    def __init__(self, root="./data", split="test", download=True):
        self.base = torchvision.datasets.OxfordIIITPet(
            root=root,
            split=split,
            target_types=["segmentation", "binary-category"],
            download=download,
        )

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        image, (trimap, species) = self.base[idx]
        trimap = np.array(trimap)

        foreground = trimap != 2  # 2 == background; 1 (pet) and 3 (border) are foreground
        ys, xs = np.where(foreground)
        box = [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]
        label = CAT_ID if species == 0 else DOG_ID

        target = {
            "boxes": torch.tensor([box], dtype=torch.float32),
            "labels": torch.tensor([label], dtype=torch.int64),
        }
        return image, target


eval_dataset = OxfordPetDetection(root="./data", split="test", download=True)
print(f"{len(eval_dataset)} images available for evaluation.")

# We only need a subset to get meaningful mAP numbers -- raise this if you have GPU time.
subset_size = 200
rng = np.random.default_rng(0)
subset_indices = rng.choice(len(eval_dataset), size=subset_size, replace=False)


In [ ]:
# Visualize a few images with their ground-truth boxes.
COCO_NAMES = {CAT_ID: "cat", DOG_ID: "dog"}

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, idx in zip(axes, subset_indices[:4]):
    image, target = eval_dataset[idx]
    image_tensor = pil_to_tensor(image)
    labels_str = [COCO_NAMES[l.item()] for l in target["labels"]]
    drawn = draw_bounding_boxes(image_tensor, target["boxes"], labels=labels_str, colors="lime", width=3)
    ax.imshow(to_pil_image(drawn))
    ax.axis("off")
plt.tight_layout()
plt.show()


## Task 2: Load a pretrained object detector

We'll use **Faster R-CNN** (ResNet-50 + FPN backbone), pretrained by `torchvision` on the full
91-class COCO dataset — which conveniently already includes `cat` (id 17) and `dog` (id 18).
As with the segmentation model in Lab 4b, this detector has never seen Oxford-IIIT Pet during
training: we're evaluating it zero-shot.

Faster R-CNN applies **non-maximum suppression** internally (see Lecture 8), so the raw
output for one image is a relatively small set of non-overlapping candidate boxes, each with a
predicted class and a confidence score — but it can still predict boxes for *any* of the 91 COCO
classes, and at low confidence. Before comparing to our ground truth, we need to keep only the
detections that are actually relevant.


In [ ]:
weights = ???  # hint: FasterRCNN_ResNet50_FPN_Weights.DEFAULT
detector = fasterrcnn_resnet50_fpn(weights=weights)
detector.eval().to(device)

preprocess = weights.transforms()


@torch.no_grad()
def run_detector(image_pil):
    '''Returns the raw prediction dict {boxes, labels, scores} for one PIL image.'''
    input_tensor = preprocess(image_pil).to(device)
    prediction = detector([input_tensor])[0]
    return {k: v.cpu() for k, v in prediction.items()}


def filter_predictions(prediction, score_threshold=???, keep_labels=(CAT_ID, DOG_ID)):
    '''Keeps only detections above `score_threshold` whose label is in `keep_labels`.'''
    keep = ???  # boolean mask: (prediction["scores"] > score_threshold) & isin(prediction["labels"], keep_labels)
    return {k: v[keep] for k, v in prediction.items()}


In [ ]:
# Quick sanity check on one image.
image, target = eval_dataset[subset_indices[0]]
raw_pred = run_detector(image)
pred = filter_predictions(raw_pred)
print("Raw detections:", len(raw_pred["boxes"]), "-> after filtering:", len(pred["boxes"]))
print(pred)


## Task 3: Metrics

### 3.1 Intersection over Union (IoU)

As in Lab 4b, IoU measures the overlap between two boxes (or masks):

$$\text{IoU}(A, B) = \frac{\text{area}(A \cap B)}{\text{area}(A \cup B)}$$

For boxes it's simple enough to implement yourself — a good exercise, and useful for a quick
sanity check on individual predictions. Fill in the gap below (a box is given as `[xmin, ymin, xmax, ymax]`).


In [ ]:
def box_iou(boxA, boxB):
    '''IoU between two boxes, each given as [xmin, ymin, xmax, ymax].'''
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    intersection = max(0, xB - xA) * max(0, yB - yA)  # 0 when the boxes don't overlap
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union = ???

    return intersection / union


# Sanity check: two identical boxes should have IoU 1.0; two disjoint boxes should have IoU 0.0.
assert box_iou([0, 0, 10, 10], [0, 0, 10, 10]) == 1.0
assert box_iou([0, 0, 10, 10], [20, 20, 30, 30]) == 0.0
print("box_iou passes the sanity checks.")


In [ ]:
# Every image in this dataset has exactly one ground-truth box (one pet, one box per image),
# so we can record the IoU of the detector's highest-confidence matching-class detection against
# it for every image (0 if there is no such detection).
best_ious = []

for idx in subset_indices:
    image, target = eval_dataset[idx]
    if len(target["boxes"]) != 1:
        continue  # defensive: shouldn't happen with this dataset, but keep the check anyway

    gt_box = target["boxes"][0].tolist()
    gt_label = target["labels"][0].item()

    pred = filter_predictions(run_detector(image))
    same_class = [i for i, l in enumerate(pred["labels"].tolist()) if l == gt_label]

    if not same_class:
        best_ious.append(0.0)
        continue

    ious = [box_iou(gt_box, pred["boxes"][i].tolist()) for i in same_class]
    best_ious.append(max(ious))

best_ious = np.array(best_ious)
print(f"Evaluated {len(best_ious)} single-pet images.")
print(f"Mean best-match IoU: {best_ious.mean():.4f}")
plt.hist(best_ious, bins=20)
plt.xlabel("Best IoU with ground truth")
plt.ylabel("No. of images")
plt.title("Per-image best-match IoU")
plt.show()


### 3.2 Mean Average Precision (mAP)

A single IoU number per image is useful for spot-checking, but it doesn't tell the full story:
it says nothing about missed detections, extra false-positive boxes, or how performance changes
as you vary the confidence threshold. **Mean Average Precision (mAP)** is the standard metric that
accounts for all of this at once, by summarizing the precision-recall trade-off (recall Lab 4a!)
across confidence thresholds and IoU thresholds, then averaging across classes.

Implementing COCO-style mAP correctly from scratch is a serious undertaking, so — just like we used
`torchmetrics` for Dice/IoU in Lab 4b — we'll use [`torchmetrics.detection.MeanAveragePrecision`](https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision),
which wraps the official `pycocotools` evaluator (the same one used to report results in most
detection papers).

`MeanAveragePrecision` expects **unfiltered** predictions (it needs the confidence scores to compute
precision/recall at many thresholds itself) and ground truth, both as lists of dicts with `boxes`
and `labels` (and `scores` for predictions).


In [ ]:
metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)

for idx in subset_indices:
    image, target = eval_dataset[idx]
    raw_pred = run_detector(image)  # NOTE: unfiltered -- MeanAveragePrecision needs the scores

    metric.update(preds=[raw_pred], target=[target])

results = metric.compute()
for key in ["map", "map_50", "map_75", "map_per_class", "classes"]:
    print(f"{key}: {results[key]}")


In [ ]:
# Visualize predictions (after filtering) vs. ground truth on a few images.
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, idx in zip(axes, subset_indices[:4]):
    image, target = eval_dataset[idx]
    pred = filter_predictions(run_detector(image))

    image_tensor = pil_to_tensor(image)
    drawn = draw_bounding_boxes(image_tensor, target["boxes"], colors="lime", width=3)
    pred_labels_str = [f"{COCO_NAMES[l.item()]} {s.item():.2f}" for l, s in zip(pred["labels"], pred["scores"])]
    drawn = draw_bounding_boxes(drawn, pred["boxes"], labels=pred_labels_str, colors="red", width=3)

    ax.imshow(to_pil_image(drawn))
    ax.axis("off")
plt.suptitle("Green = ground truth, Red = prediction")
plt.tight_layout()
plt.show()


## Discussion

- Compare `map_50` (mAP at IoU ≥ 0.5) to `map_75` (IoU ≥ 0.75) and the overall COCO-style `map`
  (averaged over IoU 0.5–0.95). If `map_50` is much higher than `map_75`, what does that tell you
  about the *localization precision* of the detector (as opposed to whether it finds the pet at all)?
- Look at `map_per_class` alongside `classes` — does the detector do noticeably better on one of
  cat/dog than the other? Any hypotheses why?
- Which images in your visualizations are hardest for the detector (multiple animals, occlusion,
  an unusually small or cropped head)?
- **Optional further exploration:** `torchvision` also ships single-stage detectors (e.g. RetinaNet,
  SSD). How do their mAP and inference speed compare to Faster R-CNN's on this same evaluation set?
  (Ties into the two-stage vs. single-stage discussion from Lecture 8.)

---

## Wrapping up the series

Across these three notebooks you've computed and interpreted six different evaluation metrics —
top-1/top-5 accuracy/error, confusion matrices and precision/recall/AP for classification; IoU and Dice for
segmentation; IoU and mAP for detection. This is exactly the kind of quantitative evaluation your
course project report is expected to include (see the "What characterizes a good report?" guidance
from the project introduction): a clear description of your dataset, the *correct* metrics for your
task, and an honest discussion of where and why your model struggles.